# Notebook 08: Dense Semantic Embedding Generation

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
While lexical algorithms (Boolean, TF-IDF, BM25) rely on exact token matching, **Dense Semantic Representations** map words, phrases, and entire documents into continuous vector spaces where semantically similar items reside close together.

In this notebook, we explore:
1. **Dense Multilingual Model**: Using `intfloat/multilingual-e5-small` (384 dimensions) with asymmetric prefixes (`query:` vs `passage:`).
2. **Vector Space Properties**: $L_2$ normalization, cosine similarity as inner products ($A \cdot B = \cos(\theta)$).
3. **Corpus Embedding Generation & Caching**: Efficient batch encoding with MD5 content hashing.
4. **Cross-Lingual Semantic Matching**: Matching Khmer queries directly against English documents without translation.
5. **Semantic Similarity Analysis**: Visualizing cosine similarities across diverse Phnom Penh cuisine categories.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter
from src.embeddings.embedding_model import EmbeddingModel
from src.embeddings.embedding_generator import EmbeddingGenerator

print(f"Project root: {PROJECT_ROOT}")

## 1. Load Restaurant Dataset and Document Corpus

In [ ]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
docs = converter.convert_all(records)
corpus = converter.get_corpus()

print(f"Loaded {len(records)} restaurant records.")
print(f"Sample Document Text [ID: {records[0]['id']}]:\n{corpus[records[0]['id']]}")

## 2. Initialize Multilingual-E5 Embedding Model

`intfloat/multilingual-e5-small` requires specific task prefixes:
- Passages (documents): Prepend `passage: `
- Queries: Prepend `query: `

All output vectors are normalized to unit length ($||v||_2 = 1.0$).

In [ ]:
model = EmbeddingModel()
print(f"Model Name : {model.model_name}")
print(f"Dimension  : {model.get_dimension()}")
print(f"Device     : {model.device}")

In [ ]:
# Test encoding and verify L2 normalization
sample_query = "romantic rooftop bar with riverside view"
query_vec = model.encode_query(sample_query, normalize_embeddings=True)

print(f"Query Vector Shape : {query_vec.shape}")
print(f"L2 Norm of Vector  : {np.linalg.norm(query_vec):.6f} (Should be 1.000000)")
print(f"First 5 components : {np.round(query_vec[:5], 4)}")

## 3. Generate or Load Cached Corpus Embeddings

`EmbeddingGenerator` caches dense embeddings in `vector_db/embeddings.npy` with an MD5 hash of the corpus.

In [ ]:
vector_dir = PROJECT_ROOT / "vector_db"
generator = EmbeddingGenerator(embedding_model=model, vector_dir=vector_dir)

embeddings, doc_ids = generator.generate_or_load(corpus)
print(f"Total Embeddings Matrix Shape: {embeddings.shape}")
print(f"Total Doc IDs Mapped         : {len(doc_ids)}")

## 4. Cross-Lingual Semantic Alignment (Khmer vs. English)

Multilingual-E5 maps semantic concepts into a shared multilingual embedding space.
Here we test whether a query written in Khmer can match English documents semantically without translation.

In [ ]:
queries = {
    "English Italian Query": "Italian pizza and pasta restaurant",
    "Khmer Italian Query"  : "ភោជនីយដ្ឋានភីហ្សា និងប៉ាស្តាអ៊ីតាលី",
    "English Khmer Food"   : "Traditional Khmer amok and local soup",
    "Khmer Food Query"     : "អាហារខ្មែរប្រពៃណី អាម៉ុក និងសម្ល"
}

encoded_queries = {label: model.encode_query(q) for label, q in queries.items()}

# Compute cosine similarity matrix between the English and Khmer queries
labels = list(queries.keys())
sim_matrix = np.zeros((len(labels), len(labels)))

for i, l1 in enumerate(labels):
    for j, l2 in enumerate(labels):
        sim_matrix[i, j] = np.dot(encoded_queries[l1], encoded_queries[l2])

sim_df = pd.DataFrame(sim_matrix, index=labels, columns=labels)
print("Cross-Lingual Query Similarity Matrix:")
sim_df.round(4)

Notice how:
- *English Italian Query* and *Khmer Italian Query* have very high similarity (~0.85+).
- *English Khmer Food* and *Khmer Food Query* have very high similarity (~0.85+).
- Cross-topic similarity between Italian food and Khmer soup remains low (~0.60).

This confirms robust multilingual semantic alignment.

## 5. Key Takeaways
1. **Dense Embeddings** capture abstract semantics, synonyms, and intent beyond surface keywords.
2. **Multilingual Alignment**: High semantic fidelity across English and Khmer without manual translation pipelines.
3. **Caching**: Precomputing the 698 document embeddings reduces startup and query overhead to zero.
4. **Next**: In **Notebook 09**, we load these vectors into a FAISS index for sub-millisecond similarity search!